# 25 · A rule over a shape, against Shapely

**Question.** Can a rule classify a quantity the geometry DERIVED, rather than a column
somebody imported — and does the answer come back attached to the shape it is about?

Three steps in one chain, each rebuilt here from Fieldwork's declaration rather than taken
on trust: the convex hull of 250 located cholera deaths, the area of that hull in
EPSG:32630, and the classification of that area against a 250,000 m² threshold.

**Why the whole chain and not the rule.** The rule is a comparison, and checking a
comparison proves nothing. What [experiment 113](https://github.com/digital-twin-ph/fieldwork/blob/main/docs/experiments/113-rules-over-geometry.md)
doubted is whether a derived quantity can reach a rule at all. Until 2026-10-14 it could
not: Fieldwork computed the hull's area, published it in the run receipt, and dropped it
before anything downstream could read it. So a disagreement in the status here is as likely
to be a disagreement about the area as about the threshold.

**Criteria fixed before running, with reasons.**

1. *The ring, vertex for vertex.* A convex hull is exact — no tolerance, no approximation —
   so any difference is a real difference.
2. *The area within 0.01%.* Both take a shoelace area over projected metres, so they should
   agree to floating point; the tolerance is for the projection library, not the arithmetic.
3. *The same status, for the same stated comparison.* A status that agreed by accident — the
   right word from the wrong numbers — would pass a check of the word alone, so the
   explanation is read for the same two figures.

In [ ]:
NEEDED = ["shapely", "pyproj", "numpy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_25_extent import run

report = run(Path.cwd() / "fixtures" / "extent-soho-001.json")
print("criterion:", report["criterion"])
print("agrees:", report["agrees"])
print(f"points hulled: {report['points']}")
print(f"hull vertices: Fieldwork {report['fieldwork']['hullVertices']}, "
      f"Shapely {report['reference']['hullVertices']}, rings agree: {report['ringsAgree']}")
print(f"area: Fieldwork {report['fieldwork']['areaSquareMetres']:.1f} m2, "
      f"Shapely {report['reference']['areaSquareMetres']:.1f} m2 "
      f"({report['areaDifferenceFraction']*100:.4f}%)")
print(f"status: Fieldwork {report['fieldwork']['status']}, "
      f"Shapely {report['reference']['status']}")
print("because:", report["fieldwork"]["explanation"])

## Dive deeper · three leaps this makes

The check above is short. What it is checking took three changes that are easy to miss when
reading the code, and each one is the difference between a rule that can run over geometry
and a rule that cannot.

### 1. A derived quantity has to be a field, not a line in a receipt

`summarize_hull` always computed the area. It put it in the operation's notes and asserted
it as `fw:hullAreaSquareMetres` in the run receipt — both of which a human reads and no
widget does. The feature it emitted carried `id`, `name`, `siteId` and `geometry`, and
nothing else.

A connection between two widgets carries a **value**, not a receipt. So a rule downstream
could see the shape and not its size, and reaching back into a receipt to find the number
is how a thin carrier becomes a thick one. The area is now attached to each polygon as
`areaSquareMetres`, by the widget that projected to get it.

**What the absence means.** A widget that worked in degrees attaches nothing. That absence
becomes `fw:areaKnown false` and a status of **Unknown** — not a zero, and not a silent
pass. A polygon whose area nobody measured is not a polygon of no area.

### 2. A decision travels on the thing it judges

A ruleset over points returned its subject whole, with a status added, which is why those
decisions can be drawn on a map. A ruleset over polygons returned `{id, name, status,
area, explanation}` and dropped the geometry — so a decision about a shape could be listed
and never drawn, clipped, summarised, or carried any further.

From `extent-review` **1.1.0** the ruleset's declared output port is `polygons`, and each
feature carries its own `status` and `explanation`. The port type lives in the ruleset
document, so this was a version of the ruleset rather than a change to the widget.

### 3. A composite can only replace a chain if it emits what the chain emitted

Fieldwork can save `summarize_hull → apply_ruleset` as one step — a composition — which
expands back into its members when it runs, one receipt each. The way to test one is to
substitute it into a worked example and see whether the example still wires up.

The John Snow outbreak-extent example feeds its hull to **two** places: the ruleset, and
the map that draws the outline. While the ruleset emitted `decisions`, a composite built
from those two widgets emitted `decisions` too, and the map had nothing to connect to. With
the output port change above, the composite emits polygons, and the substitution keeps every
edge. The leap worth noticing: **the port type decided whether a composition was possible**,
not the composition machinery.

In [ ]:
# Leap 1, demonstrated: the same rule over a polygon with no area attached.
# Unknown is a result, not a failure -- and it is what a widget working in degrees produces.
fixture = json.loads((Path.cwd() / "fixtures" / "extent-soho-001.json").read_text())
threshold = fixture["declaredParameters"]["thresholdSquareMetres"]
area = fixture["output"]["areaSquareMetres"]

for described, known in (("area attached", True), ("no area attached", False)):
    status = ("Review" if area > threshold else "NoFlag") if known else "Unknown"
    print(f"{described:>18}: fw:areaKnown {str(known).lower():5} -> {status}")

In [ ]:
# What the unrecorded threshold is worth. The hull does not move; only the line does.
# Every figure below is the same observed outbreak, classified differently.
print(f"hull area: {area:,.0f} m2 ({area/1e6:.3f} km2)\n")
print(f"{'threshold km2':>14} {'status':>8}")
for km2 in (0.05, 0.10, 0.139, 0.25, 0.50):
    print(f"{km2:14.3f} {'Review' if area > km2 * 1e6 else 'NoFlag':>8}")
print("\nThe case study records no reason for any of these, and neither does the ruleset.")

## What agreement here does not mean

That 250,000 m² is the right threshold, or that a convex hull is the right outline. Both
are the practitioner's to choose, and neither Fieldwork nor this notebook ships a basis for
either.

It also does not check the reasoner. The fixture records what EYE concluded, and this file
recomputes the same comparison in Python — a check of the derivation and the threshold, not
of the inference. Checking the rules themselves against an enumerated decision table is
[check 23](./23-rulesets-vs-decision-table.ipynb).

And a hull is an outline of **what was observed**. It encloses ground where nothing was
recorded, and says nothing at all about what lies outside it.